# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`). Every entity is referenced by its `@id`, which ensures consistent access during data operations.


In [ ]:
# List available record sets by @id
record_sets = dataset.record_sets
print("Available record sets (by @id):\n------------------------------")
for rec in record_sets:
    print(f"@id: {rec['@id']}  |  name: {rec['name']}")

if not record_sets:
    print("No record sets were listed in dataset.metadata. Please check the Croissant schema or dataset.release for updates.")
else:
    # For demonstration, select the first record set
    first_record_set_id = record_sets[0]['@id']
    print(f"\nFields for record set {first_record_set_id}:")
    fields = [f for f in dataset.fields(record_set=first_record_set_id)]
    for field in fields:
        print(f"\t@id: {field['@id']}  |  name: {field.get('name','')}")


## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All references to record sets and fields use their `@id` as shown in the overview above.

In [ ]:
# Prepare a dictionary of dataframes for all record sets
record_set_ids = [rec['@id'] for rec in record_sets]
dataframes = {}

for rsid in record_set_ids:
    records_iter = dataset.records(record_set=rsid)
    records = list(records_iter)
    if records:
        df = pd.DataFrame(records)
        dataframes[rsid] = df
        print(f"Loaded DataFrame with shape {df.shape} for record set @id: {rsid}")

# Display the first few columns and records for the first available record set
if record_set_ids and record_set_ids[0] in dataframes:
    main_record_set_id = record_set_ids[0]
    print(f"Example columns for record set {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records, normalizing numeric fields, and grouping data by key attributes. All field selections use the `@id` as identified above.

In [ ]:
from IPython.display import display
import numpy as np

# Example: Filter and normalize a numeric field (choose actual @id's from field listing above)
if 'main_record_set_id' in locals() and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    # Identify a numeric field
    # For demonstration, attempt to auto-find a float/int column
    numeric_fields = df.select_dtypes(include=['int', 'float', np.number]).columns.tolist()
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field for filtering: {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notna().sum() > 0 else 0
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a likely categorical field
        group_field_candidates = df.columns.difference([numeric_field_id])
        group_field = None
        for col in group_field_candidates:
            if df[col].dtype == 'object' or str(df[col].dtype).startswith('category'):
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean {numeric_field_id} by {group_field}:")
            display(grouped_df.head())
    else:
        print("No numeric fields found in the primary DataFrame for EDA.")
else:
    print("Main record set not available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset, always referencing fields by their `@id`.

In [ ]:
import matplotlib.pyplot as plt

if 'filtered_df' in locals() and not filtered_df.empty:
    # Histogram of the normalized numeric field
    field_norm = f"{numeric_field_id}_normalized"
    plt.figure(figsize=(8, 4))
    plt.hist(filtered_df[field_norm].dropna(), bins=20, alpha=0.7)
    plt.title(f"Distribution of Normalized {numeric_field_id}")
    plt.xlabel("Normalized Value")
    plt.ylabel("Count")
    plt.show()

    # Boxplot grouped by group_field if available
    if group_field:
        plt.figure(figsize=(10, 4))
        filtered_df.boxplot(column=numeric_field_id, by=group_field, grid=False, rot=45)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.suptitle("")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No filtered DataFrame to visualize.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* In this notebook, you learned to load, explore, and process the Mass Spectrometry Analysis dataset using the `mlcroissant` library entirely by referencing dataset entities via their `@id` fields.
* We reviewed record sets and fields, extracted records to DataFrames, and performed basic EDA including filtering, normalization, and visualization.
* For serious scientific analysis, always refer to the precise field and record set `@id` in code, as dataset structure and exact naming may evolve.
* For advanced downstream tasks, continue to leverage the full set of features offered by Croissant metadata and the `mlcroissant` library.
